In [9]:
import os
from glob import glob
import toml
from hand_tracker.anipose_yt.triangulate import triangulate
from hand_tracker.triangulation.lp2anipose import lp2anipose
from hand_tracker.anipose_yt.triangulate import process_trial as triangulate_process

In [10]:
# 
camera_views = ["To", "TL", "TR", "BL", "BR"]
lp_dir = '/home/yiting/Documents/LP_debug/20260415/litpose'
ap_dir = '/home/yiting/Documents/LP_debug/20260415/anipose'
lp_2d_dir = os.path.join(lp_dir, "video_preds")
ap_2d_dir = os.path.join(ap_dir, "pose_2d")
os.makedirs(ap_2d_dir, exist_ok = True)

# t = 'Neo_2025-11-20_08-07-35_974890'
# for c in camera_views:
#     lp_file_path = os.path.join(lp_2d_dir, t + "_cam" +  c + ".csv")
#     anipose_file_path = os.path.join(ap_2d_dir, t + "_cam" +  c + ".h5")
#     lp2anipose(lp_file_path, anipose_file_path)



In [11]:
config_file = os.path.join(ap_dir, "config.toml")
# Load config file
config = toml.load(config_file)

trials = sorted(os.listdir(ap_2d_dir))
session_name = "20260415"

trial_name = trials[0]

In [ ]:
config['triangulation']['score_threshold'] = 0.7
config['triangulation']['ransac'] = True

In [ ]:
pipeline_data_dir = config['pipeline']['data_dir']
pipeline_analysis_dir = config['pipeline']['analysis_dir']            
pipeline_pose2d = config['pipeline']['pose_2d']
pipeline_pose2d_filter = config['pipeline']['pose_2d_filter']
pipeline_pose3d = config['pipeline']['pose_3d']

pipeline_calibration = config['pipeline']['calibration_folder']
pipeline_calibration_results = config['pipeline']['calibration_results']

calib_folder = os.path.join(pipeline_analysis_dir, pipeline_calibration, pipeline_calibration_results)

if config['filter']['enabled']:
    pose2d_folder = os.path.join(pipeline_analysis_dir, session_name,
                            "anipose", pipeline_pose2d_filter, trial_name)
else:
    pose2d_folder = os.path.join(pipeline_analysis_dir, session_name,
                            "anipose", pipeline_pose2d, trial_name)


video_folder = os.path.join(pipeline_data_dir, session_name, "cameras", trial_name) 
output_fname = os.path.join(pipeline_analysis_dir, session_name, 
                            "anipose", pipeline_pose3d, trial_name + "_3d.csv")

pose_2d_files = glob(os.path.join(pose2d_folder, '*.h5'))
camera_views = config['camera']['camera_views']
fname_dict = dict(zip(sorted(camera_views), sorted(pose_2d_files)))

triangulate(config, calib_folder, video_folder, pose2d_folder,
            fname_dict, output_fname)

In [14]:
print(fname_dict)

{'BL': '/home/yiting/Documents/LP_debug/20260415/anipose/pose_2d/Neo_2025-11-20_08-07-35_974890/Neo_2025-11-20_08-07-35_974890_camBL.h5', 'BR': '/home/yiting/Documents/LP_debug/20260415/anipose/pose_2d/Neo_2025-11-20_08-07-35_974890/Neo_2025-11-20_08-07-35_974890_camBR.h5', 'TL': '/home/yiting/Documents/LP_debug/20260415/anipose/pose_2d/Neo_2025-11-20_08-07-35_974890/Neo_2025-11-20_08-07-35_974890_camTL.h5', 'TR': '/home/yiting/Documents/LP_debug/20260415/anipose/pose_2d/Neo_2025-11-20_08-07-35_974890/Neo_2025-11-20_08-07-35_974890_camTR.h5', 'To': '/home/yiting/Documents/LP_debug/20260415/anipose/pose_2d/Neo_2025-11-20_08-07-35_974890/Neo_2025-11-20_08-07-35_974890_camTo.h5'}


In [13]:
import yaml
lp_config_path = '/home/yiting/Documents/GitHub/lightning-pose/scripts/configs/config_multiview_cal.yaml'

with open(lp_config_path, 'r') as file:
    lp_config = yaml.safe_load(file)

kpt_names = lp_config['data']['keypoint_names']

In [19]:
import pandas as pd

pose3d_folder = os.path.join(pipeline_analysis_dir, session_name, "anipose", pipeline_pose3d)
for t in trials[0:2]:
    pose3d_path = os.path.join(pose3d_folder, t + "_3d.csv")
    pose3d_df = pd.read_csv(pose3d_path)
    error_columns = [col for col in pose3d_df.columns if col.endswith('_error')]
    error_df = pose3d_df[error_columns]
    mean_error = error_df.mean()
    grand_mean_error = mean_error.mean()
    error_range_across_keypoints = mean_error.max() - mean_error.min()
    
    print(t)
    print(f"Grand mean reprojection error:")
    print(f"{grand_mean_error.item():.2f} pixels")
    print(f"Min and Max mean reprojection error across keypoints:")
    print(f"{mean_error.min():.2f} pixels, {mean_error.max():.2f} pixels")
    print(f"Range of reprojection error across keypoints:")
    print(f"{error_range_across_keypoints:.2f} pixels")
    print("\n")


Neo_2025-11-20_08-07-35_974890
Grand mean reprojection error:
2.66 pixels
Min and Max mean reprojection error across keypoints:
1.64 pixels, 5.36 pixels
Range of reprojection error across keypoints:
3.72 pixels


Neo_2025-11-20_08-34-56_328042
Grand mean reprojection error:
2.33 pixels
Min and Max mean reprojection error across keypoints:
1.30 pixels, 4.26 pixels
Range of reprojection error across keypoints:
2.96 pixels




In [20]:
from aniposelib.cameras import CameraGroup
import numpy as np

calibration_file = os.path.join(calib_folder, "calibration.toml")
# 1. Load your calibration
cgroup = CameraGroup.load(calibration_file)

In [21]:
from aniposelib.utils import load_pose2d_fnames

d = load_pose2d_fnames(fname_dict, cam_names=cgroup.get_names())

score_threshold = 0.5

n_cams, n_points, n_joints, _ = d['points'].shape
points = d['points']
scores = d['scores']

bodyparts = d['bodyparts']

# remove points that are below threshold
points[scores < score_threshold] = np.nan

points_flat = points.reshape(n_cams, -1, 2)
scores_flat = scores.reshape(n_cams, -1)

p3ds_flat = cgroup.triangulate(points_flat, progress=True)
reprojerr_flat = cgroup.reprojection_error(p3ds_flat, points_flat, mean=True)

p3ds = p3ds_flat.reshape(n_points, n_joints, 3)
reprojerr = reprojerr_flat.reshape(n_points, n_joints)

100%|█████████████████████████| 25830/25830 [00:04<00:00, 6008.25it/s]


In [22]:
# Calculate Mean Reprojection Error (MRE)
mean_error = np.nanmean(reprojerr)
print(f"Mean Reprojection Error: {mean_error:.3f} pixels")

Mean Reprojection Error: 6.939 pixels
